# Fine-tune LoRA Papua AI di Colab (T4)

1. Runtime → Change runtime type → **T4 GPU**
2. Jalankan sel berurutan
3. Dataset SFT ikut di repo (`research/lora/sft_train.jsonl`)

In [ ]:
!pip install -q transformers peft datasets accelerate
!git clone --depth 1 https://github.com/KANZAPRO71/kutumbaba.git repo || true
%cd repo
!python research/lora/build_sft_dataset.py

In [ ]:
!python research/lora/train_lora_hf.py --model Qwen/Qwen2.5-0.5B-Instruct --epochs 2 --rank 16

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

base = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(base, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(base, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True)
model = PeftModel.from_pretrained(model, "research/lora/hf_adapter")

prompts = [
    "Apa arti tra?",
    "Kasi sa mop satu.",
    "Siapa yang buat app ini?",
    "Bilang dalam bahasa Papua: Saya sudah makan.",
]
sys = (
    "Kamu Papua AI, teman ngobrol Melayu Papua urban. "
    "Pakai sa/ko, tra/su, mo, toh, kah. Jangan pakai beta."
)
for p in prompts:
    messages = [{"role": "system", "content": sys}, {"role": "user", "content": p}]
    ids = tok.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to(model.device)
    out = model.generate(ids, max_new_tokens=80, temperature=0.7, top_p=0.9)
    print("=", p)
    print(tok.decode(out[0], skip_special_tokens=True))
    print()